In [1]:
# 02_masking_and_recall.ipynb -- Apply tiered masking (C0-C4) to the real-code corpus and measure masker recall, leakage, and precision against injected ground truth.
# Masker recall is the regulatory threshold variable; precision captures over-masking (false positives) on real code.
# Output paths are intentionally not printed (anonymity).

import json
import re
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Reproducibility & grayscale publication style -------------------------
SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "savefig.dpi": 600, "figure.dpi": 120, "font.size": 11,
    "axes.edgecolor": "0.2", "grid.color": "0.85", "image.cmap": "Greys",
})
GRAYS = ["0.10", "0.30", "0.50", "0.68", "0.82"]
MARKERS = ["o", "s", "^", "D", "v"]

_cwd = Path.cwd()
ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
DATA = ROOT / "data"
FIG = ROOT / "results" / "figures"
TAB = ROOT / "results" / "tables"
for d in (FIG, TAB):
    d.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)

with open(DATA / "corpus.json", encoding="utf-8") as f:
    corpus = json.load(f)
gt = pd.read_csv(DATA / "ground_truth.csv", dtype={"value": str})

# --- Sensitive detectors: regex + checksum validators ----------------------
def luhn_ok(num):
    s = re.sub(r"\D", "", num)
    if len(s) != 16:
        return False
    total = sum(int(d) for d in s[-1::-2])
    for d in s[-2::-2]:
        d2 = int(d) * 2
        total += d2 - 9 if d2 > 9 else d2
    return total % 10 == 0

def rrn_ok(num):
    d = re.sub(r"\D", "", num)
    if len(d) != 13:
        return False
    w = [2, 3, 4, 5, 6, 7, 8, 9, 2, 3, 4, 5]
    return (11 - (sum(int(d[i]) * w[i] for i in range(12)) % 11)) % 10 == int(d[12])

PATTERNS = {
    "RRN": re.compile(r"\b\d{6}-?\d{7}\b"),
    "CARD": re.compile(r"\b(?:\d{4}[- ]?){3}\d{4}\b"),
    "SECRET": re.compile(r"\b(?:sk-[A-Za-z0-9]{20,}|AKIA[A-Z0-9]{16}|ghp_[A-Za-z0-9]{30,}|[0-9a-f]{40})\b"),
    "EMAIL": re.compile(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b"),
    "PHONE": re.compile(r"\b01[016789]-?\d{3,4}-?\d{4}\b"),
    "ACCOUNT": re.compile(r"\b\d{11,14}\b"),
}
VALIDATORS = {"RRN": rrn_ok, "CARD": luhn_ok}
SENSITIVE_ORDER = ["SECRET", "EMAIL", "RRN", "CARD", "PHONE", "ACCOUNT"]

# --- Identifier / literal heuristics for higher tiers (general Java) --------
JAVA_KW = set("""abstract assert boolean break byte case catch char class const continue default do double
else enum extends final finally float for goto if implements import instanceof int interface long native
new package private protected public return short static strictfp super switch synchronized this throw
throws transient try void volatile while true false null var record yield sealed permits""".split())
COMMON = set("""String List Map Set Integer Long Double Float Boolean Object Override Exception Collection
Optional BigDecimal BigInteger LocalDate LocalDateTime Date Arrays Collections Math System out println
get set is to of value name id type data org apache fineract com java util""".split())
# Masking passes use alternation so existing placeholders are matched-and-kept before the identifier/literal
# group can touch their internals (no sentinels; avoids corrupting <ID_..>/<TYPE_..> tokens at higher tiers).
KEEP = r"<[A-Z]+_[0-9a-f]{6}>"
IDENT_PASS = re.compile(r"(" + KEEP + r")|([A-Za-z_$][A-Za-z0-9_$]*)")
STR_PASS = re.compile(r"(" + KEEP + r")|(\"[^\"]*\")")
NUM_PASS = re.compile(r"(" + KEEP + r"|\"<STR>\"|<NUM>)|(\b\d+(?:\.\d+)?\b)")

def _tok(prefix, s):
    return f"<{prefix}_{hashlib.md5(s.encode()).hexdigest()[:6]}>"

def mask_text(tier, text):
    # Returns (masked_text, id_map, counts, masked_sensitive_values).
    counts = {"sensitive": 0, "identifier": 0, "literal": 0}
    id_map = {}
    masked_vals = []
    if tier == "C0":
        return text, id_map, counts, masked_vals
    masked = text
    # Sensitive (C1+): mask longer/structured types first.
    for t in SENSITIVE_ORDER:
        def repl(m, t=t):
            val = m.group(0)
            if t in VALIDATORS and not VALIDATORS[t](val):
                return val
            counts["sensitive"] += 1
            masked_vals.append(val)
            return _tok(t, val)
        masked = PATTERNS[t].sub(repl, masked)
    if tier in ("C2", "C3", "C4"):
        def repl_id(m):
            if m.group(1):          # existing placeholder: keep intact
                return m.group(1)
            w = m.group(2)
            # Mask only compound code identifiers (camelCase / snake_case / with digits),
            # which excludes plain English words in comments and the license header.
            is_code_id = bool(re.search(r"[a-z][A-Z]", w)) or ("_" in w) or any(c.isdigit() for c in w)
            if w in JAVA_KW or w in COMMON or len(w) < 3 or not is_code_id:
                return w
            counts["identifier"] += 1
            tok = _tok("ID", w)
            id_map[tok] = w
            return tok
        masked = IDENT_PASS.sub(repl_id, masked)
    if tier in ("C3", "C4"):
        def repl_str(m):
            if m.group(1):
                return m.group(1)
            counts["literal"] += 1
            return "\"<STR>\""
        masked = STR_PASS.sub(repl_str, masked)
        def repl_num(m):
            if m.group(1):
                return m.group(1)
            counts["literal"] += 1
            return "<NUM>"
        masked = NUM_PASS.sub(repl_num, masked)
    if tier == "C4":
        id_map = {}  # irreversible: mapping discarded
    return masked, id_map, counts, masked_vals

TIERS = ["C0", "C1", "C2", "C3", "C4"]
TOKEN_TYPES = ["RRN", "CARD", "ACCOUNT", "SECRET", "EMAIL", "PHONE"]
gt_by_file = {fid: g[["type", "value"]].values.tolist() for fid, g in gt.groupby("file_id")}
gt_values = {fid: set(str(v) for _, v in rows) for fid, rows in gt_by_file.items()}

rows, type_rows = [], []
masked_corpus = {t: [] for t in TIERS}
id_maps = {}
for tier in TIERS:
    caught = {t: 0 for t in TOKEN_TYPES}
    total = {t: 0 for t in TOKEN_TYPES}
    tp = fp = 0
    for item in corpus:
        fid, text = item["file_id"], item["code"]
        masked, id_map, counts, masked_vals = mask_text(tier, text)
        masked_corpus[tier].append({"file_id": fid, "code": masked, **counts})
        if tier == "C2":
            id_maps[str(fid)] = id_map
        for typ, val in gt_by_file.get(fid, []):
            total[typ] += 1
            if tier != "C0" and str(val) not in masked:
                caught[typ] += 1
        for v in masked_vals:  # precision of sensitive masking
            if v in gt_values.get(fid, set()):
                tp += 1
            else:
                fp += 1
    tot_all = sum(total.values())
    caught_all = sum(caught.values())
    recall = caught_all / tot_all if tot_all else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 1.0
    rows.append({"tier": tier, "recall": recall, "leakage": 1 - recall, "precision": precision,
                 "sensitive_total": tot_all, "sensitive_caught": caught_all})
    for typ in TOKEN_TYPES:
        type_rows.append({"tier": tier, "type": typ,
                          "recall": caught[typ] / total[typ] if total[typ] else 0.0})

recall_df = pd.DataFrame(rows)
type_df = pd.DataFrame(type_rows)
recall_df.to_csv(TAB / "recall_by_tier.csv", index=False)
type_df.to_csv(TAB / "recall_by_type.csv", index=False)
with open(DATA / "masked_corpus.json", "w", encoding="utf-8") as f:
    json.dump(masked_corpus, f, ensure_ascii=False)
with open(DATA / "id_maps.json", "w", encoding="utf-8") as f:
    json.dump(id_maps, f, ensure_ascii=False)

# --- Figure: recall, leakage, precision across tiers (grayscale) -----------
fig, ax = plt.subplots(figsize=(5.4, 3.5))
ax.plot(recall_df["tier"], recall_df["recall"], color=GRAYS[0], marker=MARKERS[0], linewidth=1.8, label="Recall")
ax.plot(recall_df["tier"], recall_df["leakage"], color=GRAYS[2], marker=MARKERS[1], linestyle="--", linewidth=1.8, label="Leakage")
ax.plot(recall_df["tier"], recall_df["precision"], color=GRAYS[3], marker=MARKERS[2], linestyle=":", linewidth=1.8, label="Precision")
ax.set_xlabel("Masking tier")
ax.set_ylabel("Proportion")
ax.set_ylim(-0.03, 1.03)
ax.legend(frameon=False, loc="center right")
save_fig(fig, "fig_recall_leakage_by_tier")

# --- Figure: per-type recall (grayscale) -----------------------------------
fig, ax = plt.subplots(figsize=(5.6, 3.6))
for i, typ in enumerate(TOKEN_TYPES):
    sub = type_df[type_df["type"] == typ]
    ax.plot(sub["tier"], sub["recall"], color=GRAYS[i % len(GRAYS)], marker=MARKERS[i % len(MARKERS)],
            linewidth=1.5, label=typ)
ax.set_xlabel("Masking tier")
ax.set_ylabel("Recall by token type")
ax.set_ylim(-0.03, 1.03)
ax.legend(frameon=False, ncol=2, fontsize=9)
save_fig(fig, "fig_recall_by_type")

print("recall / leakage / precision by tier:")
print(recall_df[["tier", "recall", "leakage", "precision"]].to_string(index=False))


recall / leakage / precision by tier:
tier  recall  leakage  precision
  C0     0.0      1.0        1.0
  C1     1.0      0.0        1.0
  C2     1.0      0.0        1.0
  C3     1.0      0.0        1.0
  C4     1.0      0.0        1.0
